## Выводы и анализ ошибок

- Выбор финальной версии: модель выбиралась по `val_direction_accuracy` (приор.), затем `val_r2`, затем `val_rmse` — это даёт баланс между направлением движения и регрессионной точностью.
- Обоснование: лучшая версия демонстрирует стабильно выше доли верно предсказанных направлений (business metric), R2 близок к нулю (малый вклад регрессии), RMSE не сильно хуже других вариантов.
- Фиксация PRD: финальная версия помечена тегом `PRD` в MLflow и сохранена как артефакт `model`.
- Воспроизводимость: зафиксированы seed и все гиперпараметры (в логах MLflow: `sequence_length`, `hidden_size`, `num_layers`, `dropout`, `lr`, `batch_size`, `epochs`).
- Артефакты: модель, learning curves, prediction plots, CSV с топ-ошибками и robustness-варинтами залиты в MLflow и сохраняются в S3/MinIO (см. конфиг `MLFLOW_S3_ENDPOINT_URL`).
- Baseline: использован нулевой baseline (предсказание 0 change). Финальная модель превосходит baseline по direction_accuracy и часто по RMSE на тесте.
- Категории ошибок: 1) малоамплитудные движения (шум) 2) резкие события (новости вне обучающей истории) 3) лаг в тональности/сигнале 4) недостающие или искажённые признаки.
- Примеры ошибок: в артефакте `top_errors_df` сохранены 10–20 примеров с исходными признаками и объяснениями (колонки `abs_error`, `direction_match`, `news_excerpt`).
- Причины и корректировка: большинство ошибок связаны с недостающей информации (новость после свечи, тональность недостаточно репрезентативна) — корректировка ограничена; можно улучшить сбор новостей/лагирование, но это выходит за рамки модели.
- Robustness: протестированы варианты с нулевой тональностью, сдвигом тональности и шумом — результаты логируются в `robustness_df`; наблюдение: направление устойчево к небольшому шуму, абсолютная ошибка растёт при искусственном занулении тональности.
- Дальше: в `DL_Demonstration.ipynb` показан пример загрузки PRD-модели из MLflow и выполнения тестового предикта.


In [ ]:
# Пример: загрузка PRD-модели из MLflow и тестовый предикт

import os
import mlflow
import mlflow.pytorch
import mlflow.pyfunc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://localhost:5050')
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

# Поиск run с тегом PRD: если сервер недоступен, просто показываем инструкцию и не падаем
server_available = True
try:
    df_runs = mlflow.search_runs(experiment_ids=None, filter_string=None, max_results=500)
except Exception as e:
    server_available = False
    print('Не удалось подключиться к MLflow:', e)
    print('Проверь, что MLflow запущен и доступен по', MLFLOW_TRACKING_URI)
    print('Если локальный стек ещё не поднят, запусти docker compose / mlflow-сервис и повтори.')
    df_runs = pd.DataFrame()

runs = []
if server_available and not df_runs.empty:
    tag_cols = [c for c in df_runs.columns if c.startswith('tags.')]
    for _, row in df_runs.iterrows():
        for col in tag_cols:
            val = row.get(col)
            if pd.isna(val):
                continue
            tag_key = col[5:]
            if tag_key.lower() == 'prd' or 'PRD' in str(val).upper():
                runs.append({'run_id': row['run_id'], 'end_time': row.get('end_time') or row.get('start_time')})
                break

if not runs:
    print('PRD run не найден или MLflow недоступен.')
    if server_available:
        print('Проверь, что в MLflow действительно есть run с тегом PRD.')
        try:
            recent = mlflow.search_runs(experiment_ids=None, filter_string=None, order_by=["attributes.end_time DESC"], max_results=10)
            if not recent.empty:
                print('Последние 10 run-ов:')
                print(recent[[c for c in recent.columns if c == 'run_id' or c.startswith('tags.')]].to_string(index=False))
        except Exception as e:
            print('Не удалось получить список последних run-ов:', e)
    else:
        print('Сначала подними MLflow, затем повторно запусти эту ячейку.')
else:
    runs_sorted = sorted(runs, key=lambda r: r.get('end_time') or 0, reverse=True)
    run_id = runs_sorted[0]['run_id']
    print('Выбран run_id:', run_id)

    model = None
    try:
        model = mlflow.pytorch.load_model(f'runs:/{run_id}/model')
        print('Загружено через mlflow.pytorch')
    except Exception as e:
        print('pytorch load failed:', e)
        try:
            model = mlflow.pyfunc.load_model(f'runs:/{run_id}/model')
            print('Загружено через mlflow.pyfunc')
        except Exception as e2:
            print('pyfunc load failed:', e2)
            model = None

    def try_show_artifact_image(run_id, artifact_name):
        try:
            from mlflow import artifacts
            local = artifacts.download_artifacts(run_id=run_id, artifact_path=artifact_name)
        except Exception:
            try:
                local = client.download_artifacts(run_id, artifact_name)
            except Exception as e:
                print(f'Не удалось скачать артефакт {artifact_name}:', e)
                return
        try:
            if os.path.isdir(local):
                candidates = [f for f in os.listdir(local) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
                if not candidates:
                    print('Артефакт найден, но изображений не обнаружено в', local)
                    return
                local = os.path.join(local, candidates[0])
            img = plt.imread(local)
            plt.figure(figsize=(10,4))
            plt.imshow(img)
            plt.axis('off')
            plt.title(artifact_name)
            plt.show()
        except Exception as e:
            print(f'Не удалось загрузить/показать артефакт {artifact_name}:', e)

    try_show_artifact_image(run_id, 'learning_curves.png')
    try_show_artifact_image(run_id, 'prd_predictions.png')
    try_show_artifact_image(run_id, 'predictions.png')

    demo_test_csv = 'mlflow/demo_sber_test.csv'
    if not os.path.exists(demo_test_csv):
        print(f"Файл с тестом не найден: {demo_test_csv}\nСохраните подготовленный DataFrame туда или измените путь в переменной `demo_test_csv`.")
    elif model is None:
        print('Модель не загрузилась, поэтому тестовый предикт не выполняется.')
    else:
        df_test = pd.read_csv(demo_test_csv)
        X = df_test.drop(columns=[c for c in df_test.columns if c.lower() in ('target', 'target_price_change', 'y')], errors='ignore')
        try:
            preds = model.predict(X)
        except Exception:
            import torch
            model.eval()
            with torch.no_grad():
                tensor = torch.tensor(X.values, dtype=torch.float32)
                out = model(tensor)
                try:
                    preds = out.detach().cpu().numpy()
                except Exception:
                    preds = np.array(out)
        df_test['prd_predict'] = preds
        out_path = 'mlflow/prd_demo_predictions.csv'
        df_test.to_csv(out_path, index=False)
        print('Saved predictions to', out_path)
        if 'target_price_change' in df_test.columns:
            plt.figure(figsize=(10,4))
            plt.plot(df_test['target_price_change'].values, label='target')
            plt.plot(df_test['prd_predict'].values, label='predict')
            plt.legend()
            plt.title('PRD model predictions vs target')
            plt.show()


Не удалось подключиться к MLflow: API request to http://localhost:5000/api/2.0/mlflow/runs/search failed with exception HTTPConnectionPool(host='localhost', port=5000): Max retries exceeded with url: /api/2.0/mlflow/runs/search (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000002469EF6D2E0>: Failed to establish a new connection: [WinError 10061] Подключение не установлено, т.к. конечный компьютер отверг запрос на подключение'))
Проверь, что MLflow запущен и доступен по http://localhost:5000
Если локальный стек ещё не поднят, запусти docker compose / mlflow-сервис и повтори.
PRD run не найден или MLflow недоступен.
Сначала подними MLflow, затем повторно запусти эту ячейку.


## How to run

- Установите `MLFLOW_TRACKING_URI` если MLflow не на `http://localhost:5000`.
- Убедитесь, что существует run, помеченный тегом `PRD` (или вручную укажите `run_id`).
- Подготовьте файл `mlflow/demo_sber_test.csv` с признаками, которые модель ожидает, и опционально колонкой `target_price_change`.
- Запустите ячейки по порядку: модель загрузится и выполнится тестовый предикт.
